# MVP - Data Engineering

## Camada gold

Aqui, criamos e documentamos as tabelas da camada gold. Depois, adicionamos os constraints Primary Key/Foreign Key. É importante notar que estes são apenas informativos - não são verificados no Databricks.

Implementamos também um checksum de validação, comparando a soma de sales_value na fato_vendas com a soma original em order_items para garantir que os JOINs não duplicaram registros na tabela fato.

Finalmente, fizemos uma verificação da qualidade dos dados.


In [0]:
%run /Users/cyntia_invernizzi@hotmail.com/_DataEng_PUCRIO/common

In [0]:
# 1. Python Libraries

from pyspark.sql.functions import col, avg as spark_avg, max as spark_max, percentile_approx


In [0]:
%sql
-- Estabelecendo o catálogo usado no MVP (SQL)

USE CATALOG mvp_pucrio;

In [0]:
# Estabelecendo as variáveis para catálogo e schema para serem usadas neste Notebook em Pyspark.

catalog = "mvp_pucrio"

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%sql
USE SCHEMA gold;

A dimensão `dim_leads` (abaixo) consolida todos os leads qualificados de marketing (MQLs), convertidos ou não. O `LEFT JOIN` com `closed_deals` preserva os leads que não foram convertidos, preenchendo os campos relacionados ao seller com "unknown" via `COALESCE`. Isso permite análises de funil completo — desde o primeiro contato até a conversão — sem perder leads que não avançaram no funil.

In [0]:
%sql
-- Criando dim_leads consolidada

CREATE OR REPLACE TABLE dim_leads AS
SELECT 
    mql.mql_id,
    COALESCE(cd.seller_id, 'unknown') AS seller_id,
    mql.origin,
    mql.first_contact_date,
    CAST(cd.won_date AS DATE) AS won_date,
    DATEDIFF(CAST(cd.won_date AS DATE), mql.first_contact_date) AS sales_cycle
FROM silver.marketing_qualified_leads mql
LEFT JOIN silver.closed_deals cd
    ON mql.mql_id = cd.mql_id;

In [0]:
%sql
-- Documentando dim_leads

COMMENT ON TABLE dim_leads IS 
'Dimensão consolidada de TODOS os leads qualificados de marketing (MQLs) - tanto convertidos quanto não convertidos. Combina dados de marketing_qualified_leads com closed_deals via LEFT JOIN para facilitar análises de funil completo. Leads não convertidos terão won_date e sales_cycle como NULL. seller_id terão valor "unknown". Uso: análises de funil de conversão, taxa de conversão por canal, ciclo de vendas, performance de segmentos.';

COMMENT ON COLUMN dim_leads.mql_id IS 
'Primary Key - ID único do lead qualificado de marketing. Tipo: STRING. Justificativa: mql_id é sempre único (cada lead é um ponto de entrada distinto), enquanto seller_id pode repetir se um seller veio de múltiplos leads.';

COMMENT ON COLUMN dim_leads.seller_id IS 
'ID do seller resultante da conversão. Tipo: STRING. "unknown" para leads não convertidos. Corresponde a dim_sellers.seller_id quando diferente de "unknown".';

COMMENT ON COLUMN dim_leads.origin IS 
'Canal de origem do lead (organic_search, paid_search, social, email, direct ou "unknown"). Tipo: STRING. Usado para análise de ROI por canal e taxa de conversão por origem.';

COMMENT ON COLUMN dim_leads.first_contact_date IS 
'Data do primeiro contato com o lead. Tipo: DATE. Usado para calcular sales_cycle e análises de coorte temporal.';

COMMENT ON COLUMN dim_leads.won_date IS 
'Data de conversão (fechamento do negócio). Tipo: DATE. NULL para leads não convertidos. Usado para calcular sales_cycle e análises de conversão ao longo do tempo.';

COMMENT ON COLUMN dim_leads.sales_cycle IS 
'Dias entre primeiro contato e conversão. Tipo: INT. Fórmula: DATEDIFF(won_date, first_contact_date). NULL para leads não convertidos. Métrica chave para eficiência do funil de vendas.';

In [0]:
%sql
-- Constraint PK - dim_leads

ALTER TABLE dim_leads 
ALTER COLUMN mql_id SET NOT NULL;

ALTER TABLE dim_leads 
ADD CONSTRAINT pk_dim_leads PRIMARY KEY (mql_id);

In [0]:
%sql
-- Verificando dim_leads

SELECT * FROM dim_leads LIMIT 10;

In [0]:
%sql
-- Criando dim_produtos

CREATE OR REPLACE TABLE dim_produtos AS
SELECT DISTINCT
    product_id,
    product_category_name AS category
FROM silver.products;

In [0]:
%sql
-- Documentando dim_produtos

COMMENT ON TABLE dim_produtos IS 
'Dimensão de produtos disponíveis na plataforma. Origem: silver.products. Uso: análise de receita por categoria, mix de produtos. IMPORTANTE: Esta dimensão contém apenas atributos descritivos (category). Métricas como sales_value estão na fato_vendas.';

COMMENT ON COLUMN dim_produtos.product_id IS 
'Primary Key - ID único do produto. Tipo: STRING.';

COMMENT ON COLUMN dim_produtos.category IS 
'Categoria do produto (eletrônicos, cama_mesa_banho, beleza_saúde, brinquedos, automotivo, etc ou "unknown"). Tipo: STRING. Origem: silver.products.product_category_name.';

In [0]:
%sql
-- Constraint PK - dim_produtos

ALTER TABLE dim_produtos 
ALTER COLUMN product_id SET NOT NULL;

ALTER TABLE dim_produtos 
ADD CONSTRAINT pk_dim_produtos PRIMARY KEY (product_id);

In [0]:
%sql
-- Verificando dim_produtos

SELECT * FROM dim_produtos LIMIT 10;

In [0]:
%sql
-- Criando dim_dates

CREATE OR REPLACE TABLE dim_dates AS
SELECT DISTINCT
    order_date,
    YEAR(order_date) AS year,
    MONTH(order_date) AS month,
    QUARTER(order_date) AS quarter,
    DAYOFWEEK(order_date) AS day_of_week
FROM silver.orders;

In [0]:
%sql
-- Documentando dim_dates

COMMENT ON TABLE dim_dates IS 
'Dimensão temporal para análises por período. Origem: DISTINCT order_purchase_timestamp de silver.orders. Uso: análises de tendência, sazonalidade, performance por trimestre/mês/dia da semana.';

COMMENT ON COLUMN dim_dates.order_date IS 
'Primary Key - Data da transação. Tipo: DATE. Formato: YYYY-MM-DD. Valores: todas as datas únicas de compras.';

COMMENT ON COLUMN dim_dates.year IS 
'Ano extraído da order_date. Tipo: INT. Valores: 2016-2018. Fórmula: YEAR(order_purchase_timestamp).';

COMMENT ON COLUMN dim_dates.month IS 
'Mês extraído da order_date. Tipo: INT. Valores: 1-12. Fórmula: MONTH(order_purchase_timestamp). Usado para análise de sazonalidade.';

COMMENT ON COLUMN dim_dates.quarter IS 
'Trimestre do ano. Tipo: INT. Valores: 1-4. Fórmula: QUARTER(order_purchase_timestamp). Usado para relatórios trimestrais.';

COMMENT ON COLUMN dim_dates.day_of_week IS 
'Dia da semana. Tipo: INT. Valores: 1-7 (1=Domingo, 7=Sábado). Fórmula: DAYOFWEEK(order_purchase_timestamp). Usado para análise de padrões semanais.';

In [0]:
%sql
-- Constraint PK - dim_dates

ALTER TABLE dim_dates 
ALTER COLUMN order_date SET NOT NULL;

ALTER TABLE dim_dates 
ADD CONSTRAINT pk_dim_dates PRIMARY KEY (order_date);

In [0]:
%sql
-- Verificando dim_dates

SELECT * FROM dim_dates LIMIT 10;

In [0]:
%sql
-- Criando fato_vendas

CREATE OR REPLACE TABLE fato_vendas AS
SELECT 
    o.order_id,
    COALESCE(c.mql_id, "unknown") AS mql_id,
    oi.seller_id,
    cust.customer_state,
    oi.product_id,
    CAST(o.order_purchase_timestamp AS DATE) AS order_date,
    COALESCE(o.order_status, 'unknown') AS order_status,
    oi.quantity,
    oi.sales_value,
    ROUND(oi.sales_value * 0.10, 2) AS commission
FROM silver.orders o
INNER JOIN silver.order_items oi 
    ON o.order_id = oi.order_id
LEFT JOIN silver.closed_deals c 
    ON oi.seller_id = c.seller_id
LEFT JOIN silver.customers cust
    ON o.customer_id = cust.customer_id;

A tabela fato `fato_vendas` é o centro do modelo estrela. Sua criação envolve três joins:

1. `INNER JOIN` entre `orders` e `order_items` — cada linha da fato representa um item vendido dentro de uma ordem.
2. `LEFT JOIN` com `closed_deals` **no nível do seller** (por `seller_id`, não por `order_id`) — isso significa que toda venda de um seller herda o mesmo `mql_id`, ou seja, o canal de aquisição é atribuído ao seller, não à venda individual. Sellers que não vieram de um lead rastreado recebem "unknown".
3. `LEFT JOIN` com `customers` (por `customer_id`) — traz `customer_state` para a fato, representando o local onde o produto do seller foi vendido.

A comissao da plataforma e fixa em 10% sobre o valor da venda, calculada como ROUND(sales_value * 0.10, 2).

In [0]:
%sql
-- Documentando fato_vendas

COMMENT ON TABLE fato_vendas IS 
'Tabela FATO do modelo star schema. Cada linha representa uma transação de venda (item vendido). Origem: JOIN entre silver.orders e silver.order_items, com LEFT JOIN para silver.closed_deals. Contém métricas agregáveis (quantity, sales_value, comission) e chaves estrangeiras para todas as dimensões.';

COMMENT ON COLUMN fato_vendas.order_id IS 
'Primary Key - ID único da ordem de venda. Tipo: STRING. Origem: silver.orders.order_id.';

COMMENT ON COLUMN fato_vendas.mql_id IS 
'Foreign Key → dim_leads.mql_id. Identificador do lead qualificado de marketing associado ao seller desta venda. Tipo: STRING. Origem: silver.closed_deals.mql_id via LEFT JOIN por seller_id ("unknown" se seller não veio de lead rastreado).
ATENÇÃO: atribuição feita no nível do seller (via closed_deals.seller_id), não da venda individual — toda venda de um seller herda o canal de aquisição daquele seller.';

COMMENT ON COLUMN fato_vendas.seller_id IS 
'Identificador do vendedor que realizou a venda. Tipo: STRING. Origem: silver.order_items.seller_id. SEM NULOS.';

COMMENT ON COLUMN fato_vendas.customer_state IS 
'Estado (UF) onde o produto do seller foi vendido. Tipo: STRING. Valores: SP, RJ, MG, etc ou "unknown". Origem: silver.customers.customer_state via LEFT JOIN por customer_id.';

COMMENT ON COLUMN fato_vendas.product_id IS 
'Foreign Key → dim_produtos.product_id. Identificador do produto vendido. Tipo: STRING. Origem: silver.order_items.product_id. SEM NULOS.';

COMMENT ON COLUMN fato_vendas.order_date IS 
'Foreign Key → dim_dates.order_date. Data da transação para análises temporais. Tipo: DATE. Formato: YYYY-MM-DD. Origem: silver.orders.order_date. SEM NULOS.';

COMMENT ON COLUMN fato_vendas.order_status IS 
'Status do pedido (delivered, shipped, canceled, processing, etc ou "unknown"). Tipo: STRING. Origem: COALESCE(silver.orders.order_status, "unknown"). SEM NULOS.';

COMMENT ON COLUMN fato_vendas.quantity IS 
'MÉTRICA - Quantidade de itens do mesmo produto na ordem. Tipo: INT. Valores: >= 1. Origem: silver.order_items.quantity (COUNT de linhas agrupadas por order_id + product_id + seller_id). SEM NULOS. Agregável: SUM, AVG.';

COMMENT ON COLUMN fato_vendas.sales_value IS 
'MÉTRICA - Valor total vendido nesta transação (em BRL). Tipo: DECIMAL. Valores: >= 0. Origem: silver.order_items.sales_value. SEM NULOS. Agregável: SUM, AVG.';

COMMENT ON COLUMN fato_vendas.commission IS 
'MÉTRICA - Comissão da plataforma (10% do sales_value). Tipo: DECIMAL. Valores: >= 0. Fórmula: ROUND(sales_value * 0.10, 2). SEM NULOS. Agregável: SUM, AVG.';

In [0]:
%sql
-- Constraints PK e FKs - fato_vendas

ALTER TABLE fato_vendas 
ALTER COLUMN order_id SET NOT NULL;

ALTER TABLE fato_vendas 
ALTER COLUMN product_id SET NOT NULL;

ALTER TABLE fato_vendas 
ALTER COLUMN seller_id SET NOT NULL;

ALTER TABLE fato_vendas 
ALTER COLUMN order_date SET NOT NULL;

ALTER TABLE fato_vendas 
ADD CONSTRAINT pk_fato_vendas PRIMARY KEY (order_id, product_id, seller_id);

ALTER TABLE fato_vendas 
ADD CONSTRAINT fk_fato_leads FOREIGN KEY (mql_id) 
REFERENCES dim_leads(mql_id);

ALTER TABLE fato_vendas 
ADD CONSTRAINT fk_fato_produtos FOREIGN KEY (product_id) 
REFERENCES dim_produtos(product_id);

ALTER TABLE fato_vendas 
ADD CONSTRAINT fk_fato_dates FOREIGN KEY (order_date) 
REFERENCES dim_dates(order_date);

In [0]:
%sql
-- Verificando fato_vendas

SELECT * FROM fato_vendas LIMIT 10;

In [0]:
%sql
-- Checksum

SELECT ROUND(SUM(f.sales_value) - SUM(oi.sales_value),2) AS checksum
FROM silver.order_items oi 
JOIN fato_vendas f ON f.order_id = oi.order_id;

## Verificando a qualidade dos dados

Aqui, verificamos a completude, consistência, unicidade e acurácia dos dados, assim como se há outliers que possam distorcer análises estatísticas. Com isso, podemos identificar se há a necessidade de algum ajuste antes de prosseguirmos com as análises finais.

In [0]:
def check_temporal_accuracy(df, table_name, last_date):
    """Verifica se datas fazem sentido no contexto do negócio."""
    problems = 0
    
    # 1. Datas no futuro (impossível)
    date_cols = ["order_date", "first_contact_date", "won_date"]
    for date_col in date_cols:
        if date_col in df.columns:
            future_dates = df.filter(col(date_col) > last_date)
            count = future_dates.count()
            if count > 0:
                problems = 1
                print(f"\nQuantidade de linhas com {date_col} no futuro: {count}")
                future_dates.limit(10).show(truncate=False)
    
    # 2. Won_date deve ser >= first_contact_date (para leads convertidos)
    if table_name == "dim_leads":
        # Filtrar apenas leads convertidos (com won_date)
        joined = df.filter(col("won_date").isNotNull())
        invalid_dates = joined.filter(
            (col("won_date").isNotNull()) & 
            (col("first_contact_date").isNotNull()) &
            (col("won_date") < col("first_contact_date"))
        )
        count = invalid_dates.count()
        if count > 0:
            problems += count
            print(f"\nQuantidade de linhas onde won_date < first_contact_date: {count}")
            invalid_dates.select("mql_id", "first_contact_date", "won_date").limit(10).show(truncate=False)
    
    return problems

In [0]:
def detect_outliers(df, table_name):
    """Detecta outliers em preços unitários usando método IQR (Interquartile Range)."""
    problems = 0
    
    if table_name == "fato_vendas":
        df = df.withColumn("unit_price", col("sales_value") / col("quantity"))
        
        product_stats = df.groupBy("product_id").agg(
            percentile_approx("unit_price", 0.25).alias("q1"),
            percentile_approx("unit_price", 0.75).alias("q3"),
            percentile_approx("unit_price", 0.50).alias("median"),
            spark_avg("unit_price").alias("avg_price")
        ).withColumns({
            "lower_bound": col("q1") - 1.5 * (col("q3") - col("q1")),
            "upper_bound": col("q3") + 1.5 * (col("q3") - col("q1"))
        })
        
        produtos = spark.table("dim_produtos")
        joined = df.join(product_stats, "product_id", "left") \
                   .join(produtos, "product_id", "left")
        outliers = joined.filter(
            (col("unit_price") < col("lower_bound")) | (col("unit_price") > col("upper_bound"))
        )
        
        outlier_count = outliers.count()

        if outlier_count > 0:
            problems = 1
            print(f"\nOutliers detectados (por produto): {outlier_count} linhas")
            outliers.select(
                "product_id", "category", "order_id", "sales_value", "quantity", "unit_price", "avg_price",
                "median", "lower_bound", "upper_bound"
            ).orderBy(col("unit_price").desc()).limit(10).show(truncate=False)
        
    return problems

In [0]:
# Análise COMPLETA de qualidade dos dados (Gold)

nulls_dict = dict()
duplicates_set = set()

print("\n" + "="*80)
print("QUALIDADE DE DADOS - GOLD LAYER")

problems = 0
schema = f"{catalog}.gold"
table_names = schema_tables(schema)

for name in table_names:
    full_name = f"{schema}.{name}"
    df = spark.table(full_name)

    print("="*80)
    print(f"\nVerificando tabela: {name}")

    num_nulls, problems, nulls_dict = find_nulls(df, name, problems, nulls_dict)
    
    problems, duplicates_set = find_duplicates(df, "gold", name, num_nulls, problems, duplicates_set)
    
    if name in ("dim_leads", "dim_converted_leads", "fato_vendas"):
        last_date = spark.table("dim_dates") \
    .agg(spark_max("order_date")).collect()[0][0]
        problems += check_temporal_accuracy(df, name, last_date)
    if name == "fato_vendas":
        problems += detect_outliers(df, name)

if problems == 0:
    print(f"\n🎉 Nenhuma tabela com problemas detectados!")

##### Comentários

Resultados da verificação de qualidade (camada Gold):

- Completude: nenhuma tabela apresentou valores nulos nas colunas do modelo. A limpeza feita na camada Silver (valores ausentes preenchidos com "unknown") garantiu completude até a camada final.

- Consistência: validada em dois níveis — (1) integridade referencial garantida pelas constraints de FK (`fk_fato_leads`, `fk_fato_produtos`, `fk_fato_dates`) e PK composta em `fato_vendas`; (2) checksum entre `silver.order_items` e `fato_vendas` confirmou que os joins não alteraram o valor total de vendas. A consistência de valores categóricos (estados, status de pedido) já foi verificada na camada Silver e chega à Gold por herança, sem transformações que introduzam novos valores.

- Unicidade: Não há linhas duplicadas ou violação da chave composta primária composta (order_id + product_id + seller_id) na tabela fato_vendas.

- Acurácia temporal: dim_leads apresentou 17 registros de leads convertidos com won_date posterior à última data de venda registrada no dataset, e 1 registro com won_date anterior ao first_contact_date (inconsistência lógica, já que um negócio não pode ser fechado antes do primeiro contato). Optei por manter esses registros no dataset e documentar a limitação, já que representam menos de 0,1% da base de leads e não afetam as métricas de vendas (fato_vendas) — apenas análises específicas de ciclo de vendas que usem esses casos pontuais.

- Outliers: 4.195 linhas (aproximadamente 4% de fato_vendas) têm preço unitário fora do intervalo IQR esperado para o respectivo produto. Ao inspecionar os casos de maior valor, eles correspondem a categorias coerentes com preços altos (eletrônicos, relógios, informática, ferramentas de construção), com quantidade unitária normal (quantity=1) — sugerindo variação legítima de preço (versões/modelos diferentes do mesmo product_id, ou mudança de preço ao longo do tempo) e não erro de digitação. Optei por não remover essas linhas, mas registrar a decisão aqui. 